In [12]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

import numpy as np
import tensorflow as tf
import csv
import os
from PIL import Image

In [29]:
def getImgAndLab(file_path, folder_path):
    
    names = []
    annot = []

    with open(file_path, newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            annot.append(row)

    for entry_name in os.listdir(folder_path):
        full_path = os.path.join(folder_path, entry_name)
        if(not entry_name == "_annotations.csv"):
            names.append(entry_name)

    images = []
    labels = []

    for i in range(len(annot)):
        for j in range(len(names)):
            try:
                index = annot[i].index(names[j])

                path = folder_path + annot[i][0]
                original_image = Image.open(path)
                box = (int(annot[i][4]), int(annot[i][5]), int(annot[i][6]), int(annot[i][7]))
                cropped_image = original_image.crop(box)
                images.append(cropped_image)
                labels.append(annot[i][3])
                #print(cropped_image)
            except ValueError:
                continue

        target_size  = (120, 120)
    processed_imgs = []

    for img in images:
        img_resized = img.resize(target_size)
        img_rgb = img_resized.convert('RGB')
        processed_imgs.append(img_rgb)

    images_np = np.array(processed_imgs, dtype=np.float32) / 255.0

    unique_labels = sorted(list(set(labels)))
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    labels_as_integers = [label_to_int[label] for label in labels]

    labels_np = np.array(labels_as_integers)

    return images_np, labels_np


In [35]:
train_images, train_labels = getImgAndLab("./data/train/_annotations.csv", "./data/train/")
test_images, test_labels = getImgAndLab("./data/test/_annotations.csv", "./data/test/")
validation_images, validation_labels = getImgAndLab("./data/valid/_annotations.csv", "./data/valid/")


model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer="RMSprop", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=10, validation_data=(validation_images, validation_labels))
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print ('Test loss: {}, Test accuracy: {}'.format(test_loss, test_accuracy))

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 90ms/step - accuracy: 0.7473 - loss: 0.7418 - val_accuracy: 0.8739 - val_loss: 0.3874
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 89ms/step - accuracy: 0.8488 - loss: 0.4477 - val_accuracy: 0.8813 - val_loss: 0.4126
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.8853 - loss: 0.3308 - val_accuracy: 0.9044 - val_loss: 0.2895
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 94ms/step - accuracy: 0.9175 - loss: 0.2438 - val_accuracy: 0.9349 - val_loss: 0.2261
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 94ms/step - accuracy: 0.9336 - loss: 0.2001 - val_accuracy: 0.9068 - val_loss: 0.2614
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 95ms/step - accuracy: 0.9479 - loss: 0.1480 - val_accuracy: 0.9357 - val_loss: 0.2083
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.9595 - loss: 0.1203 - val_accuracy: 0.9472 - val_loss: 0.1874
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.9656 - loss: 0.1079 - 